# PathMNIST histopathology classification

This notebook runs the repository's version-controlled training and evaluation commands. Select **Runtime → Change runtime type → T4 GPU** before starting. Model selection uses the official validation split; the test split is loaded only by the evaluation cells.

In [ ]:
REPO_URL = "https://github.com/ingrid-burrowes/pathmnist-histopathology.git"
!git clone {REPO_URL}
%cd /content/pathmnist-histopathology
!python -m pip install -q -e .

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before training.")
print("GPU:", torch.cuda.get_device_name(0))

## 1. Train and evaluate the custom CNN

This is the independently written baseline. Training never constructs a test loader.

In [ ]:
!pathmnist-train --config configs/cnn.yaml

In [ ]:
!pathmnist-evaluate --checkpoint outputs/cnn/best_checkpoint.pt

In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display

cnn_metrics = json.loads(Path("outputs/cnn/test/metrics.json").read_text())
cnn_metrics["test_metrics"]

In [ ]:
display(Image("outputs/cnn/test/confusion_matrix.png"))
display(Image("outputs/cnn/test/representative_errors.png"))

## 2. Train and evaluate ResNet-18

Run this after reviewing the CNN output. It fine-tunes ImageNet-pretrained weights and selects the best epoch independently using validation macro-F1.

In [ ]:
!pathmnist-train --config configs/resnet18.yaml

In [ ]:
resnet_checkpoint = "outputs/resnet18/best_checkpoint.pt"
!pathmnist-evaluate --checkpoint {resnet_checkpoint}
!pathmnist-gradcam --checkpoint {resnet_checkpoint} --split test --index 42

In [ ]:
resnet_metrics = json.loads(Path("outputs/resnet18/test/metrics.json").read_text())
print("CNN:", cnn_metrics["test_metrics"])
print("ResNet-18:", resnet_metrics["test_metrics"])
display(Image("outputs/resnet18/test/confusion_matrix.png"))
display(Image("outputs/resnet18/test/representative_errors.png"))
display(Image("outputs/resnet18/gradcam_test_42.png"))

## 3. Download the evidence bundle

Keep the checkpoints out of Git. The ZIP contains configs, histories, metrics, predictions, and figures for updating the README.

In [ ]:
from google.colab import files

!zip -qr pathmnist-results.zip outputs -x '*.pt'
files.download("pathmnist-results.zip")